# Wind & Solar Capacity Predictor

Target => val_geracaoverificada

In [36]:
%pip install duckdb numpy pandas scikit-learn pyarrow xgboost


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
import gc
import duckdb
import pyarrow
import numpy as np
import pandas as pd
from os import listdir
from pathlib import Path
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.neighbors import BallTree
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Agregando os dados extraidos da [ONS](https://dados.ons.org.br/dataset/fator-capacidade-2):

In [38]:
df = pd.concat(
    [pd.read_parquet(f"./database/{file}") for file in listdir("./database") if file.endswith('.parquet')],
    ignore_index=True
)

Ajustando a tipagem das colunas do DF de acordo com a documentação:

In [39]:
float_columns = [
    "val_latitudesecoletora",
    "val_longitudesecoletora",
    "val_latitudepontoconexao",
    "val_longitudepontoconexao",
    "val_geracaoverificada",
    "val_capacidadeinstalada",
    "val_fatorcapacidade"
]

df[float_columns] = df[float_columns].apply(pd.to_numeric, errors="coerce")

df["din_instante"] = pd.to_datetime(df["din_instante"])

df = df.drop(['nom_subsistema', 'cod_pontoconexao', 'ceg', 'nom_estado', 'val_fatorcapacidade', 'val_geracaoprogramada'],axis=1)

In [40]:
# print(df.sort_values(by="din_instante").iloc[0]) # Encontra o registro mais antigo
ano_alvo = 2016

Lê os dados metereológicos extraídos do [INMET](https://portal.inmet.gov.br/dadoshistoricos) a partir de 2009 até 2026:

In [41]:
def _num(valor):
    """Converte '1159,54' -> 1159.54; qualquer valor inválido vira NaN."""
    return pd.to_numeric(str(valor).replace(",", "."), errors="coerce")

def ler_inmet(caminho):
    # 1) Metadados (8 primeiras linhas)
    meta = {}
    with open(caminho, encoding="latin-1") as f:
        for _ in range(8):
            chave, _, valor = f.readline().strip().partition(";")
            meta[chave.rstrip(":")] = valor.strip()

    # 2) Dados
    df = pd.read_csv(
        caminho,
        sep=";",
        skiprows=8,
        decimal=",",
        encoding="latin-1",
        na_values=["-9999"],
        index_col=False,
    )
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    df["REGIAO"] = meta.get("REGIÃO")
    df["UF"] = meta.get("UF")
    df["ESTACAO"] = meta.get("ESTAÇÃO")
    df["CODIGO_WMO"] = meta.get("CODIGO (WMO)")
    df["LATITUDE"] = _num(meta.get("LATITUDE"))
    df["LONGITUDE"] = _num(meta.get("LONGITUDE"))
    df["ALTITUDE"] = _num(meta.get("ALTITUDE"))
    df["DATA_FUNDACAO"] = pd.to_datetime(
        meta.get("DATA DE FUNDAÇÃO (YYYY-MM-DD)"), errors="coerce"
    )

    return df

pasta = Path(f"./database/INMET/{ano_alvo}")

df_weather = pd.concat(
    [ler_inmet(pasta / f) for f in listdir(pasta) if f.lower().endswith(".csv")],
    ignore_index=True,
)

In [42]:
def padronizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nomes de colunas que mudaram ao longo dos anos no INMET.

    `rename` ignora chaves ausentes, então não depende de cortes de ano.
    """
    return df.rename(columns={
        # formato até 2018
        "DATA (YYYY-MM-DD)": "DATA",
        "HORA (UTC)": "HORA",
        # formato a partir de 2019
        "Data": "DATA",
        "Hora UTC": "HORA",
        # radiação (KJ vs Kj)
        "RADIACAO GLOBAL (KJ/m²)": "RADIACAO GLOBAL (Kj/m²)",
    })


FLOAT_COLUMNS = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
    "RADIACAO GLOBAL (Kj/m²)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
    "LATITUDE",
    "LONGITUDE",
    "ALTITUDE",
]
CATEGORY_COLUMNS = ["REGIAO", "UF", "ESTACAO", "CODIGO_WMO"]

df_weather = padronizar_colunas(df_weather)

df_weather[FLOAT_COLUMNS] = df_weather[FLOAT_COLUMNS].apply(pd.to_numeric, errors="coerce")

hora = (
    df_weather["HORA"]
    .astype(str)
    .str.replace(" UTC", "", regex=False)
    .str.replace(":", "", regex=False)
    .str.zfill(4)
)
data = df_weather["DATA"].astype(str).str.replace("/", "-", regex=False)  # aceita 2019/01/01 e 2019-01-01

df_weather["DATETIME"] = pd.to_datetime(
    data + " " + hora,
    format="%Y-%m-%d %H%M",
    errors="coerce",
    utc=True,
)
df_weather["DATA_FUNDACAO"] = pd.to_datetime(df_weather["DATA_FUNDACAO"], errors="coerce")

df_weather = df_weather.drop(columns=["DATA", "HORA"])
df_weather[CATEGORY_COLUMNS] = df_weather[CATEGORY_COLUMNS].astype("category")

Join dos Datasets:

In [43]:
# O INMET aqui é só de 20XX, então filtra o df para o mesmo ano
df_anual = df[df["din_instante"].dt.year == ano_alvo].copy()

# Garbage collector
del df
gc.collect()

0

In [44]:
# ------------------------------------------------------------
# Prepara o weather
# ------------------------------------------------------------
weather = df_weather.copy()

weather["CODIGO_WMO"] = weather["CODIGO_WMO"].astype(str)
weather = weather.drop_duplicates(["CODIGO_WMO", "DATETIME"])

# Garbage collector
del df_weather
gc.collect()

0

In [45]:
# ------------------------------------------------------------
# 1) Usina -> estação mais próxima
# ------------------------------------------------------------
plantas = (
    df_anual[["id_ons", "val_latitudesecoletora", "val_longitudesecoletora"]]
    .dropna()
    .drop_duplicates("id_ons")
    .reset_index(drop=True)
)

estacoes = (
    weather[["CODIGO_WMO", "LATITUDE", "LONGITUDE"]]
    .dropna()
    .drop_duplicates("CODIGO_WMO")
    .reset_index(drop=True)
)

tree = BallTree(
    np.radians(estacoes[["LATITUDE", "LONGITUDE"]].to_numpy()),
    metric="haversine",
)
dist, idx = tree.query(
    np.radians(plantas[["val_latitudesecoletora", "val_longitudesecoletora"]].to_numpy()),
    k=1,
)

plantas["CODIGO_WMO"] = estacoes["CODIGO_WMO"].to_numpy()[idx[:, 0]]
plantas["dist_estacao_km"] = dist[:, 0] * 6371  # raio da Terra em km

# ------------------------------------------------------------
# 2) Alinha o tempo (ONS = horário de Brasília, INMET = UTC)
# ------------------------------------------------------------
df_anual["DATETIME"] = (
    (df_anual["din_instante"] + pd.Timedelta(hours=3))
    .dt.tz_localize("UTC")
    .astype("datetime64[ns, UTC]")
)
weather["DATETIME"] = weather["DATETIME"].astype("datetime64[ns, UTC]")

# ------------------------------------------------------------
# 3) Join
# ------------------------------------------------------------
df_anual = df_anual.merge(
    plantas[["id_ons", "CODIGO_WMO", "dist_estacao_km"]],
    on="id_ons",
    how="left",
)

weather = weather.rename(columns={
    "LATITUDE": "LAT_ESTACAO",
    "LONGITUDE": "LON_ESTACAO",
    "ALTITUDE": "ALT_ESTACAO",
})

df_final = df_anual.merge(
    weather,
    on=["CODIGO_WMO", "DATETIME"],
    how="left",
    validate="many_to_one",
)

In [46]:
del df_anual
del weather
gc.collect()

0

In [47]:
df_final = df_final.drop(['val_latitudepontoconexao', 'val_longitudepontoconexao', 'LAT_ESTACAO', 'LON_ESTACAO', 'id_subsistema', 'id_estado', 'CODIGO_WMO'],axis=1)

In [48]:
print(df_final.iloc[67])

nom_pontoconexao                                           IGAPORA II - 230 kV (B)
nom_localizacao                                                           Interior
val_latitudesecoletora                                                  -14.128731
val_longitudesecoletora                                                 -42.681442
nom_modalidadeoperacao                                          Conjunto de Usinas
nom_tipousina                                                               Eólica
nom_usina_conjunto                                                    Conj. Morrão
id_ons                                                                   CJU_BAMOR
din_instante                                                   2016-01-01 01:00:00
val_geracaoverificada                                                       41.962
val_capacidadeinstalada                                                      117.6
DATETIME                                                 2016-01-01 04:00:00+00:00
dist

Consulta SQL 1)

In [49]:
df_final.to_parquet("datatemp.parquet", index=False)
df_final.to_csv("datatemp.csv", index=False)

con = duckdb.connect()
con.execute(f"""
    CREATE TABLE corridas AS
    SELECT * FROM read_parquet ('datatemp.parquet')
""")

# con.sql("SELECT COUNT(*) FROM corridas").show()

# con.sql("""
#     SELECT column_name
#     FROM information_schema.columns
#     WHERE table_name = 'corridas'
# """).show()

In [50]:
pd.set_option("display.max_columns", None)

df_final

,nom_pontoconexao,nom_localizacao,val_latitudesecoletora,val_longitudesecoletora,nom_modalidadeoperacao,nom_tipousina,nom_usina_conjunto,id_ons,din_instante,val_geracaoverificada,val_capacidadeinstalada,DATETIME,dist_estacao_km,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",REGIAO,UF,ESTACAO,ALT_ESTACAO,DATA_FUNDACAO
0,IGAPORA II - 69 kV (A),Interior,-14.208409,-42.636807,Conjunto de Usinas,Eólica,Conj. Alvorada,CJU_BAALV,2016-01-01 00:00:00,8.362,38.40,2016-01-01 03:00:00+00:00,12.141646,0.0,948.8,949.2,948.8,NaN,27.8,13.0,29.3,27.7,13.0,12.0,40.0,35.0,40.0,300.0,2.8,1.1,NE,BA,GUANAMBI,882.0,2008-04-24
1,IGAPORA II - 230 kV (B),Interior,-14.102794,-42.609369,Conjunto de Usinas,Eólica,Conj. Araçás,CJU_BAARA,2016-01-01 00:00:00,27.315,167.70,2016-01-01 03:00:00+00:00,19.107304,0.0,948.8,949.2,948.8,NaN,27.8,13.0,29.3,27.7,13.0,12.0,40.0,35.0,40.0,300.0,2.8,1.1,NE,BA,GUANAMBI,882.0,2008-04-24
2,PINDAI II - 230 kV (A),Interior,-14.468988,-42.582935,Conjunto de Usinas,Eólica,Conj. Caetité,CJU_BACTE,2016-01-01 00:00:00,20.804,58.80,2016-01-01 03:00:00+00:00,34.111781,0.0,948.8,949.2,948.8,NaN,27.8,13.0,29.3,27.7,13.0,12.0,40.0,35.0,40.0,300.0,2.8,1.1,NE,BA,GUANAMBI,882.0,2008-04-24
3,IGAPORA II - 230 kV (B),Interior,-14.186292,-42.493356,Conjunto de Usinas,Eólica,Conj. Caetité 123,CJU_BACAE,2016-01-01 00:00:00,12.604,90.00,2016-01-01 03:00:00+00:00,27.711880,0.0,948.8,949.2,948.8,NaN,27.8,13.0,29.3,27.7,13.0,12.0,40.0,35.0,40.0,300.0,2.8,1.1,NE,BA,GUANAMBI,882.0,2008-04-24
4,IGAPORA II - 230 kV (B),Interior,-14.138050,-42.478206,Conjunto de Usinas,Eólica,Conj. Caetité A,CJU_BACTA,2016-01-01 00:00:00,7.356,54.40,2016-01-01 03:00:00+00:00,30.260622,0.0,948.8,949.2,948.8,NaN,27.8,13.0,29.3,27.7,13.0,12.0,40.0,35.0,40.0,300.0,2.8,1.1,NE,BA,GUANAMBI,882.0,2008-04-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558288,S V PALMAR 2 - 525 kV (A),,-33.473520,-53.302226,Conjunto de Usinas,Eólica,Conj. Santa Vitória do Palmar,CJU_RSSVPA,2016-12-31 23:00:00,0.000,582.79,2017-01-01 02:00:00+00:00,30.496742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
558289,OSORIO 2 - 69 kV (A),,-30.079059,-50.172056,Tipo II-B,Eólica,Elebrás Cidreira 1,RSUECI,2016-12-31 23:00:00,7.103,70.00,2017-01-01 02:00:00+00:00,8.483786,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
558290,ATLANTIDA 2 - 69 kV (A),,-29.776727,-50.073596,Tipo II-B,Eólica,Xangri-lá,RSUEXG,2016-12-31 23:00:00,3.653,31.70,2017-01-01 02:00:00+00:00,26.581491,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
558291,AQUIBATA - 138 kV (A),,-26.624110,-51.713252,Conjunto de Usinas,Eólica,Conj. Água Doce,CJU_SCCEAD,2016-12-31 23:00:00,3.187,129.00,2017-01-01 02:00:00+00:00,43.627566,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


Consulta SQL 2)

Consulta SQL 3)

Consulta SQL 4)

Consulta SQL 5)

# Modelagem:

In [51]:
df_modelo = (
    df_final
    .dropna(subset=['val_geracaoverificada'])
    .sort_values(['din_instante', 'id_ons'], kind='stable')
    .reset_index(drop=True)
)

In [52]:
del df_final
gc.collect()

0

In [53]:
df_modelo['hora'] = df_modelo['din_instante'].dt.hour
df_modelo['dia_semana'] = df_modelo['din_instante'].dt.dayofweek
df_modelo['idade_estacao_anos'] = (
    (df_modelo['din_instante'].dt.tz_localize(None)
     - df_modelo['DATA_FUNDACAO'].dt.tz_localize(None)).dt.days / 365.25
)

y = df_modelo['val_geracaoverificada']

X = df_modelo.drop(columns=[
    'val_geracaoverificada',
    'din_instante', 'DATETIME', 'DATA_FUNDACAO',
    'nom_tipousina',
    'id_ons',
])

X_train, y_train = X, y

In [54]:
N_FOLDS = 4
PCT_TESTE = 0.10 

datas_tr = pd.Series(pd.to_datetime(df_modelo['din_instante']).to_numpy())
gap_cv = pd.Timedelta(hours=24)

instantes = pd.to_datetime(np.sort(datas_tr.unique()))
n = len(instantes)
bordas = [k * n // N_FOLDS for k in range(N_FOLDS + 1)]

cv_splits = []
for k in range(N_FOLDS):
    bloco = instantes[bordas[k]:bordas[k + 1]]
    ini = bloco[0]
    fim = bloco[-1] + pd.Timedelta(seconds=1)
    ini_teste = bloco[int(len(bloco) * (1 - PCT_TESTE))]

    idx_tr = np.where((datas_tr >= ini) & (datas_tr < ini_teste - gap_cv))[0]
    idx_te = np.where((datas_tr >= ini_teste) & (datas_tr < fim))[0]
    assert len(idx_tr) > 0 and len(idx_te) > 0, f"Fold vazio no bloco iniciado em {ini}"
    cv_splits.append((idx_tr, idx_te))

print(f"{len(cv_splits)} folds:")
for i, (tr, te) in enumerate(cv_splits, 1):
    print(f"  fold {i}: treino {datas_tr[tr].min():%Y-%m-%d} -> {datas_tr[tr].max():%Y-%m-%d} ({len(tr):,}) | "
          f"teste {datas_tr[te].min():%Y-%m-%d} -> {datas_tr[te].max():%Y-%m-%d} ({len(te):,})")

4 folds:
  fold 1: treino 2016-01-01 -> 2016-03-22 (114,773) | teste 2016-03-23 -> 2016-04-01 (12,991)
  fold 2: treino 2016-04-01 -> 2016-06-21 (119,812) | teste 2016-06-22 -> 2016-07-01 (14,080)
  fold 3: treino 2016-07-01 -> 2016-09-21 (126,286) | teste 2016-09-22 -> 2016-10-01 (14,651)
  fold 4: treino 2016-10-01 -> 2016-12-21 (133,624) | teste 2016-12-22 -> 2016-12-31 (15,620)


## Baseline:

## Modelos Obrigatórios: Random Forest e SVR

### Random Forest:

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols),
])

pipeline = Pipeline([
    ('prep', preprocess),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1)),
])

param_grid = {
    'rf__n_estimators':      [200], # [100, 150, 200, 220]
    'rf__max_features':      ['sqrt'], # [sqrt, log2, 1.0]
    'rf__bootstrap':         [True],
    'rf__criterion':         ['squared_error'], # ['squared_error', 'absolute_error', 'poisson']
    'rf__max_depth':         [20], # [10, 15, 20, 22]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=4,
    verbose=1
)

In [56]:
grid_search.fit(X_train, y_train)
print(f"Melhores parâmetros: {grid_search.best_params_}")

Fitting 4 folds for each of 1 candidates, totalling 4 fits
Melhores parâmetros: {'rf__bootstrap': True, 'rf__criterion': 'squared_error', 'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__n_estimators': 200}


In [57]:
# ------------------------------------------------------------
# Resultado do CV
# ------------------------------------------------------------
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"RMSE médio no CV (treino): {-grid_search.best_score_:.4f}")
print(f"Ano alvo: {ano_alvo}\n")

# ------------------------------------------------------------
# Predição fora da amostra: refaz cada fold com os melhores parâmetros
# e junta os testes dos 5 folds (não há hold-out final)
# ------------------------------------------------------------
yp, yt, cp, yb = [], [], [], []
for tr, te in cv_splits:
    m = clone(grid_search.best_estimator_)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    yp.append(np.clip(m.predict(X_train.iloc[te]), 0, None))   # geração não é negativa
    yt.append(y_train.iloc[te].to_numpy())
    cp.append(X_train.iloc[te]['val_capacidadeinstalada'].to_numpy())
    yb.append(np.full(len(te), y_train.iloc[tr].mean()))       # baseline: média do treino do fold

y_pred = np.concatenate(yp)
y_true = np.concatenate(yt)
cap = np.concatenate(cp)

# ------------------------------------------------------------
# Métricas base
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

erro_abs = np.abs(y_true - y_pred)
mask_cap = cap > 0
erro_pct_cap = np.where(mask_cap, erro_abs / np.where(mask_cap, cap, 1) * 100, np.nan)

nmae = np.nanmean(erro_pct_cap)
wape = erro_abs.sum() / y_true.sum() * 100

# Baseline ingênuo: prever sempre a média do treino
y_base = np.concatenate(yb)
rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
mae_base = mean_absolute_error(y_true, y_base)

# ------------------------------------------------------------
# Métricas intuitivas
# ------------------------------------------------------------
# 1) Confiabilidade geral: quanto do volume gerado o modelo acerta
confiabilidade = max(0.0, 100 - wape)

# 2) Diferença percentual total: previsto vs real (soma de tudo)
total_real = y_true.sum()
total_prev = y_pred.sum()
dif_total_pct = (total_prev - total_real) / total_real * 100

# 3) Diferença percentual média por amostra (só onde o real > 0, evita divisão por zero)
mask_real = y_true > 0
dif_pct_amostra = np.abs(y_pred[mask_real] - y_true[mask_real]) / y_true[mask_real] * 100
dif_pct_mediana = np.median(dif_pct_amostra)

# 4) Taxa de acerto por faixa de erro (em % da capacidade instalada)
def taxa_acerto(tol_pct):
    return np.nanmean(erro_pct_cap <= tol_pct) * 100

# 5) Classificação textual da confiabilidade
def classificar(c):
    if c >= 90: return "EXCELENTE"
    if c >= 80: return "MUITO BOA"
    if c >= 70: return "BOA"
    if c >= 60: return "REGULAR"
    return "BAIXA"

# 6) Tendência: o modelo superestima ou subestima?
vies = "superestima" if dif_total_pct > 0 else "subestima"

# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------
print("=" * 60)
print(f"  RELATÓRIO DE DESEMPENHO DO MODELO — {len(y_true):,} amostras de teste")
print("=" * 60)

print(f"\n>> CONFIABILIDADE GERAL: {confiabilidade:.1f}%  [{classificar(confiabilidade)}]")
print(f"   O modelo acerta ~{confiabilidade:.0f}% do volume de energia gerada.")

print("\n>> PREVISTO vs REAL")
print(f"   Geração real total:     {total_real:>14,.0f} MW")
print(f"   Geração prevista total: {total_prev:>14,.0f} MW")
print(f"   Diferença total:        {dif_total_pct:>+13.2f}%  (o modelo {vies} levemente)")
print(f"   Erro típico por amostra: {dif_pct_mediana:.1f}% (mediana)")

print("\n>> ERRO EM MW")
print(f"   Em média, a previsão erra por {mae:.1f} MW (baseline ingênuo: {mae_base:.1f} MW)")
print(f"   Erro médio equivale a {nmae:.1f}% da capacidade instalada")

print("\n>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)")
for tol in (0, 0.1, 1, 5, 10, 15, 20):
    t = taxa_acerto(tol)
    barra = "█" * int(t / 5) + "░" * (20 - int(t / 5))
    print(f"   até {tol:>3}%:  {barra}  {t:5.1f}%")

print("\n>> COMPARAÇÃO COM A MÉDIA")
print(f"   Melhora no erro (RMSE): {(1 - rmse / rmse_base) * 100:.1f}% melhor que o baseline")
print(f"   R² = {r2:.3f}  → o modelo explica {r2 * 100:.1f}% da variação da geração")

print("\n" + "=" * 60)

# ------------------------------------------------------------
# Salva as predições
# ------------------------------------------------------------
np.savetxt('conjunto_teste.predict', y_pred, fmt='%.3f')

Melhores parâmetros: {'rf__bootstrap': True, 'rf__criterion': 'squared_error', 'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__n_estimators': 200}
RMSE médio no CV (treino): 26.3397
Ano alvo: 2016

  RELATÓRIO DE DESEMPENHO DO MODELO — 57,342 amostras de teste

>> CONFIABILIDADE GERAL: 65.4%  [REGULAR]
   O modelo acerta ~65% do volume de energia gerada.

>> PREVISTO vs REAL
   Geração real total:          2,879,191 MW
   Geração prevista total:      2,728,555 MW
   Diferença total:                -5.23%  (o modelo subestima levemente)
   Erro típico por amostra: 28.1% (mediana)

>> ERRO EM MW
   Em média, a previsão erra por 17.4 MW (baseline ingênuo: 33.7 MW)
   Erro médio equivale a 15.6% da capacidade instalada

>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)
   até   0%:  ░░░░░░░░░░░░░░░░░░░░    0.3%
   até 0.1%:  ░░░░░░░░░░░░░░░░░░░░    0.7%
   até   1%:  ░░░░░░░░░░░░░░░░░░░░    4.9%
   até   5%:  ████░░░░░░░░░░░░░░░░   22.3%
   até  10%:  ████████░░░░░░░░

### SVR:

In [58]:
cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),   # SVR não aceita NaN
        ('scaler', StandardScaler()),
    ]), num_cols),
])

pipeline = Pipeline([
    ('prep', preprocess),
    ('svr', SVR()),
])

model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler(),
)

param_grid = {
    'regressor__svr__kernel':  ['rbf'], # ['linear', 'rbf', 'poly', 'sigmoid', 'precomputed']
    'regressor__svr__C':       [1], # [1, 10, 100]
    'regressor__svr__epsilon': [0.1], # [0.01, 0.1, 0.5]
    'regressor__svr__gamma':   ['scale'], # ['scale', 0.01, 0.1]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search.fit(X_train, y_train)
print(f"Melhores parâmetros: {grid_search.best_params_}")

Fitting 4 folds for each of 1 candidates, totalling 4 fits


In [ ]:
# ------------------------------------------------------------
# Resultado do CV
# ------------------------------------------------------------
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"RMSE médio no CV (treino): {-grid_search.best_score_:.4f}")
print(f"Ano alvo: {ano_alvo}\n")

# ------------------------------------------------------------
# Predição fora da amostra: ref  # fração final de cada bloco que vira testeaz cada fold com os melhores parâmetros
# e junta os testes dos 5 folds (não há hold-out final)
# ------------------------------------------------------------
yp, yt, cp, yb = [], [], [], []
for tr, te in cv_splits:
    m = clone(grid_search.best_estimator_)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    yp.append(np.clip(m.predict(X_train.iloc[te]), 0, None))   # geração não é negativa
    yt.append(y_train.iloc[te].to_numpy())
    cp.append(X_train.iloc[te]['val_capacidadeinstalada'].to_numpy())
    yb.append(np.full(len(te), y_train.iloc[tr].mean()))       # baseline: média do treino do fold

y_pred = np.concatenate(yp)
y_true = np.concatenate(yt)
cap = np.concatenate(cp)

# ------------------------------------------------------------
# Métricas base
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

erro_abs = np.abs(y_true - y_pred)
mask_cap = cap > 0
erro_pct_cap = np.where(mask_cap, erro_abs / np.where(mask_cap, cap, 1) * 100, np.nan)

nmae = np.nanmean(erro_pct_cap)
wape = erro_abs.sum() / y_true.sum() * 100

# Baseline ingênuo: prever sempre a média do treino
y_base = np.concatenate(yb)
rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
mae_base = mean_absolute_error(y_true, y_base)

# ------------------------------------------------------------
# Métricas intuitivas
# ------------------------------------------------------------
# 1) Confiabilidade geral: quanto do volume gerado o modelo acerta
confiabilidade = max(0.0, 100 - wape)

# 2) Diferença percentual total: previsto vs real (soma de tudo)
total_real = y_true.sum()
total_prev = y_pred.sum()
dif_total_pct = (total_prev - total_real) / total_real * 100

# 3) Diferença percentual média por amostra (só onde o real > 0, evita divisão por zero)
mask_real = y_true > 0
dif_pct_amostra = np.abs(y_pred[mask_real] - y_true[mask_real]) / y_true[mask_real] * 100
dif_pct_mediana = np.median(dif_pct_amostra)

# 4) Taxa de acerto por faixa de erro (em % da capacidade instalada)
def taxa_acerto(tol_pct):
    return np.nanmean(erro_pct_cap <= tol_pct) * 100

# 5) Classificação textual da confiabilidade
def classificar(c):
    if c >= 90: return "EXCELENTE"
    if c >= 80: return "MUITO BOA"
    if c >= 70: return "BOA"
    if c >= 60: return "REGULAR"
    return "BAIXA"

# 6) Tendência: o modelo superestima ou subestima?
vies = "superestima" if dif_total_pct > 0 else "subestima"

# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------
print("=" * 60)
print(f"  RELATÓRIO DE DESEMPENHO DO MODELO — {len(y_true):,} amostras de teste")
print("=" * 60)

print(f"\n>> CONFIABILIDADE GERAL: {confiabilidade:.1f}%  [{classificar(confiabilidade)}]")
print(f"   O modelo acerta ~{confiabilidade:.0f}% do volume de energia gerada.")

print("\n>> PREVISTO vs REAL")
print(f"   Geração real total:     {total_real:>14,.0f} MW")
print(f"   Geração prevista total: {total_prev:>14,.0f} MW")
print(f"   Diferença total:        {dif_total_pct:>+13.2f}%  (o modelo {vies} levemente)")
print(f"   Erro típico por amostra: {dif_pct_mediana:.1f}% (mediana)")

print("\n>> ERRO EM MW")
print(f"   Em média, a previsão erra por {mae:.1f} MW (baseline ingênuo: {mae_base:.1f} MW)")
print(f"   Erro médio equivale a {nmae:.1f}% da capacidade instalada")

print("\n>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)")
for tol in (0, 0.1, 1, 5, 10, 15, 20):
    t = taxa_acerto(tol)
    barra = "█" * int(t / 5) + "░" * (20 - int(t / 5))
    print(f"   até {tol:>3}%:  {barra}  {t:5.1f}%")

print("\n>> COMPARAÇÃO COM A MÉDIA")
print(f"   Melhora no erro (RMSE): {(1 - rmse / rmse_base) * 100:.1f}% melhor que o baseline")
print(f"   R² = {r2:.3f}  → o modelo explica {r2 * 100:.1f}% da variação da geração")

print("\n" + "=" * 60)

# ------------------------------------------------------------
# Salva as predições
# ------------------------------------------------------------
np.savetxt('conjunto_teste.predict', y_pred, fmt='%.3f')

Melhores parâmetros: {'regressor__svr__C': 1, 'regressor__svr__epsilon': 0.1, 'regressor__svr__gamma': 'scale', 'regressor__svr__kernel': 'rbf'}
RMSE médio no CV (treino): 10.7372
Ano alvo: 2012

  RELATÓRIO DE DESEMPENHO DO MODELO — 4,400 amostras de teste

>> CONFIABILIDADE GERAL: 69.3%  [REGULAR]
   O modelo acerta ~69% do volume de energia gerada.

>> PREVISTO vs REAL
   Geração real total:            106,387 MW
   Geração prevista total:        101,763 MW
   Diferença total:                -4.35%  (o modelo subestima levemente)
   Erro típico por amostra: 29.4% (mediana)

>> ERRO EM MW
   Em média, a previsão erra por 7.4 MW (baseline ingênuo: 16.2 MW)
   Erro médio equivale a 10.5% da capacidade instalada

>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)
   até   0%:  ░░░░░░░░░░░░░░░░░░░░    0.1%
   até 0.1%:  ░░░░░░░░░░░░░░░░░░░░    1.1%
   até   1%:  █░░░░░░░░░░░░░░░░░░░    9.6%
   até   5%:  ███████░░░░░░░░░░░░░   38.4%
   até  10%:  ████████████░░░░░░░░   60.

## Gradient Boosting:

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols),
])

pipeline = Pipeline([
    ('prep', preprocess),   
    ('xgb', XGBRegressor(
        objective='reg:squarederror',
        tree_method='hist',
        random_state=42,
        n_jobs=-1,
    )),
])

param_grid = {
    'xgb__n_estimators':     [200], # [200, 500], 
    'xgb__learning_rate':    [0.03], # [0.03, 0.1],
    'xgb__max_depth':        [6], # [4, 6, 8],
    'xgb__subsample':        [0.7], # [0.7, 1.0],
    'xgb__colsample_bytree': [0.8], # [0.8, 1.0],
    'xgb__min_child_weight': [1], # [1, 5, 10],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_splits,
    scoring='neg_root_mean_squared_error',
    n_jobs=1,
    verbose=1,
    error_score='raise',
)

In [ ]:
grid_search.fit(X_train, y_train)
print(f"Melhores parâmetros: {grid_search.best_params_}")

Fitting 4 folds for each of 144 candidates, totalling 576 fits
Melhores parâmetros: {'xgb__colsample_bytree': 0.8, 'xgb__learning_rate': 0.03, 'xgb__max_depth': 6, 'xgb__min_child_weight': 1, 'xgb__n_estimators': 200, 'xgb__subsample': 0.7}


In [ ]:
# ------------------------------------------------------------
# Resultado do CV
# ------------------------------------------------------------
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"RMSE médio no CV (treino): {-grid_search.best_score_:.4f}")
print(f"Ano alvo: {ano_alvo}\n")

# ------------------------------------------------------------
# Predição fora da amostra: ref  # fração final de cada bloco que vira testeaz cada fold com os melhores parâmetros
# e junta os testes dos 5 folds (não há hold-out final)
# ------------------------------------------------------------
yp, yt, cp, yb = [], [], [], []
for tr, te in cv_splits:
    m = clone(grid_search.best_estimator_)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    yp.append(np.clip(m.predict(X_train.iloc[te]), 0, None))   # geração não é negativa
    yt.append(y_train.iloc[te].to_numpy())
    cp.append(X_train.iloc[te]['val_capacidadeinstalada'].to_numpy())
    yb.append(np.full(len(te), y_train.iloc[tr].mean()))       # baseline: média do treino do fold

y_pred = np.concatenate(yp)
y_true = np.concatenate(yt)
cap = np.concatenate(cp)

# ------------------------------------------------------------
# Métricas base
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

erro_abs = np.abs(y_true - y_pred)
mask_cap = cap > 0
erro_pct_cap = np.where(mask_cap, erro_abs / np.where(mask_cap, cap, 1) * 100, np.nan)

nmae = np.nanmean(erro_pct_cap)
wape = erro_abs.sum() / y_true.sum() * 100

# Baseline ingênuo: prever sempre a média do treino
y_base = np.concatenate(yb)
rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
mae_base = mean_absolute_error(y_true, y_base)

# ------------------------------------------------------------
# Métricas intuitivas
# ------------------------------------------------------------
# 1) Confiabilidade geral: quanto do volume gerado o modelo acerta
confiabilidade = max(0.0, 100 - wape)

# 2) Diferença percentual total: previsto vs real (soma de tudo)
total_real = y_true.sum()
total_prev = y_pred.sum()
dif_total_pct = (total_prev - total_real) / total_real * 100

# 3) Diferença percentual média por amostra (só onde o real > 0, evita divisão por zero)
mask_real = y_true > 0
dif_pct_amostra = np.abs(y_pred[mask_real] - y_true[mask_real]) / y_true[mask_real] * 100
dif_pct_mediana = np.median(dif_pct_amostra)

# 4) Taxa de acerto por faixa de erro (em % da capacidade instalada)
def taxa_acerto(tol_pct):
    return np.nanmean(erro_pct_cap <= tol_pct) * 100

# 5) Classificação textual da confiabilidade
def classificar(c):
    if c >= 90: return "EXCELENTE"
    if c >= 80: return "MUITO BOA"
    if c >= 70: return "BOA"
    if c >= 60: return "REGULAR"
    return "BAIXA"

# 6) Tendência: o modelo superestima ou subestima?
vies = "superestima" if dif_total_pct > 0 else "subestima"

# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------
print("=" * 60)
print(f"  RELATÓRIO DE DESEMPENHO DO MODELO — {len(y_true):,} amostras de teste")
print("=" * 60)

print(f"\n>> CONFIABILIDADE GERAL: {confiabilidade:.1f}%  [{classificar(confiabilidade)}]")
print(f"   O modelo acerta ~{confiabilidade:.0f}% do volume de energia gerada.")

print("\n>> PREVISTO vs REAL")
print(f"   Geração real total:     {total_real:>14,.0f} MW")
print(f"   Geração prevista total: {total_prev:>14,.0f} MW")
print(f"   Diferença total:        {dif_total_pct:>+13.2f}%  (o modelo {vies} levemente)")
print(f"   Erro típico por amostra: {dif_pct_mediana:.1f}% (mediana)")

print("\n>> ERRO EM MW")
print(f"   Em média, a previsão erra por {mae:.1f} MW (baseline ingênuo: {mae_base:.1f} MW)")
print(f"   Erro médio equivale a {nmae:.1f}% da capacidade instalada")

print("\n>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)")
for tol in (0, 0.1, 1, 5, 10, 15, 20):
    t = taxa_acerto(tol)
    barra = "█" * int(t / 5) + "░" * (20 - int(t / 5))
    print(f"   até {tol:>3}%:  {barra}  {t:5.1f}%")

print("\n>> COMPARAÇÃO COM A MÉDIA")
print(f"   Melhora no erro (RMSE): {(1 - rmse / rmse_base) * 100:.1f}% melhor que o baseline")
print(f"   R² = {r2:.3f}  → o modelo explica {r2 * 100:.1f}% da variação da geração")

print("\n" + "=" * 60)

# ------------------------------------------------------------
# Salva as predições
# ------------------------------------------------------------
np.savetxt('conjunto_teste.predict', y_pred, fmt='%.3f')

Melhores parâmetros: {'xgb__colsample_bytree': 0.8, 'xgb__learning_rate': 0.03, 'xgb__max_depth': 6, 'xgb__min_child_weight': 1, 'xgb__n_estimators': 200, 'xgb__subsample': 0.7}
RMSE médio no CV (treino): 10.1688
Ano alvo: 2012

  RELATÓRIO DE DESEMPENHO DO MODELO — 4,400 amostras de teste

>> CONFIABILIDADE GERAL: 70.5%  [BOA]
   O modelo acerta ~71% do volume de energia gerada.

>> PREVISTO vs REAL
   Geração real total:            106,387 MW
   Geração prevista total:         98,581 MW
   Diferença total:                -7.34%  (o modelo subestima levemente)
   Erro típico por amostra: 27.9% (mediana)

>> ERRO EM MW
   Em média, a previsão erra por 7.1 MW (baseline ingênuo: 16.2 MW)
   Erro médio equivale a 10.2% da capacidade instalada

>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)
   até   0%:  ░░░░░░░░░░░░░░░░░░░░    0.0%
   até 0.1%:  ░░░░░░░░░░░░░░░░░░░░    1.0%
   até   1%:  ██░░░░░░░░░░░░░░░░░░   10.1%
   até   5%:  ███████░░░░░░░░░░░░░   38.5%
   até  10%

## TabPFN: